# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ameen740/Internship_Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 4: The Freshness Multiplier: The paper reports that recently refreshed mature pages had higher observed health and impressions than older pages that had not been refreshed recently, including a reported 52× impression difference in the 365+ page comparison. My methodology question is: how were “growth,” “decline,” and “refreshed” labels defined, and does the comparison account for the possibility that pages chosen for refreshing were already stronger or more valuable? The paper presents this as a pattern rather than proof of causation, so I would treat it as measured evidence of an association rather than proof that refreshing alone caused the increase.

Finding 5: Reader Engagement and Search Visibility Move Together: The paper reports that pages with higher scroll and reader engagement had higher health scores, with a 16.1-point difference between the strongest and weakest engagement groups. My methodology question is: how were the engagement groups and health-score outcome defined, and does the validation design separate engagement from the possibility that stronger search positions or better-performing pages naturally receive more engagement? I would therefore interpret this as an observed relationship rather than evidence that engagement directly causes better search performance.



In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-09 - Question 1
# This cell checks that the two selected paper findings are documented.

paper_findings = {
    "finding_4": "The Freshness Multiplier",
    "finding_5": "Reader Engagement and Search Visibility"
}

for name, finding in paper_findings.items():
    assert finding, f"{name} is missing"

print("Selected paper findings:")
for name, finding in paper_findings.items():
    print("-", finding)

print("\nMethodology audit focus:")
print("- Finding 4: check how freshness/growth labels are defined and whether confounding is addressed.")
print("- Finding 5: check how engagement groups are defined and whether the relationship supports a causal claim.")


Selected paper findings:
- The Freshness Multiplier
- Reader Engagement and Search Visibility

Methodology audit focus:
- Finding 4: check how freshness/growth labels are defined and whether confounding is addressed.
- Finding 5: check how engagement groups are defined and whether the relationship supports a causal claim.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before: In Week 5, I evaluated the Logistic Regression model using a client-grouped split, keeping records from the same client together so that a client did not appear in both the training and test sets. The measured Week-5 result was 0.8952 precision and 0.992887 accuracy.

After: For this validation audit, I re-ran the same model using a time-aware split, training on earlier observations and testing on later observations. This provides a different check on how sensitive the measured performance is to the validation design. I treat the difference between the two results as directional evidence about model stability rather than as proof that the model will generalize to every future client or reporting period

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-09 - Question 2
# Before: Week-5 client-grouped validation
# After: time-aware validation

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, accuracy_score


# --------------------------------------------------
# Prepare the same Week-5 modeling data
# --------------------------------------------------

model_df = df.copy()

# Week-5 CTR calculation
model_df["ctr_pct"] = (
    100.0 * model_df["gsc_clicks"] /
    model_df["gsc_impressions"].replace(0, np.nan)
)

# Week-5 baseline target
model_df["baseline_target"] = (
    (model_df["gsc_impressions"] >= 100) &
    (model_df["ctr_pct"] < 2.0)
).astype(int)

# Clean invalid values
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Week-5 model features
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
]

model_df = model_df.dropna(
    subset=features + ["client_hash_id", "report_date"]
)

print("Rows used:", len(model_df))
print("Features:", features)
print("Target: baseline_target")


# ==================================================
# BEFORE: Week-5 client-grouped split
# ==================================================

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(
        model_df,
        model_df["baseline_target"],
        groups=model_df["client_hash_id"]
    )
)

train_before = model_df.iloc[train_idx]
test_before = model_df.iloc[test_idx]

X_train_before = train_before[features]
X_test_before = test_before[features]

y_train_before = train_before["baseline_target"]
y_test_before = test_before["baseline_target"]

model_before = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model_before.fit(
    X_train_before,
    y_train_before
)

pred_before = model_before.predict(
    X_test_before
)

before_precision = precision_score(
    y_test_before,
    pred_before,
    zero_division=0
)

before_accuracy = accuracy_score(
    y_test_before,
    pred_before
)


# ==================================================
# AFTER: Time-aware split
# ==================================================

model_df = model_df.sort_values(
    "report_date"
).reset_index(drop=True)

unique_dates = sorted(
    model_df["report_date"].unique()
)

cutoff = int(len(unique_dates) * 0.80)

train_dates = unique_dates[:cutoff]
test_dates = unique_dates[cutoff:]

train_after = model_df[
    model_df["report_date"].isin(train_dates)
]

test_after = model_df[
    model_df["report_date"].isin(test_dates)
]

X_train_after = train_after[features]
X_test_after = test_after[features]

y_train_after = train_after["baseline_target"]
y_test_after = test_after["baseline_target"]

model_after = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model_after.fit(
    X_train_after,
    y_train_after
)

pred_after = model_after.predict(
    X_test_after
)

after_precision = precision_score(
    y_test_after,
    pred_after,
    zero_division=0
)

after_accuracy = accuracy_score(
    y_test_after,
    pred_after
)


# ==================================================
# BEFORE / AFTER RESULTS
# ==================================================

results = pd.DataFrame({
    "Validation": [
        "Week-5 client-grouped",
        "ML-09 time-aware"
    ],
    "Precision": [
        before_precision,
        after_precision
    ],
    "Accuracy": [
        before_accuracy,
        after_accuracy
    ]
})

print("\nBefore / After comparison:")
display(results)

print("\nMeasured changes:")
print(
    "Precision change:",
    round(after_precision - before_precision, 6)
)

print(
    "Accuracy change:",
    round(after_accuracy - before_accuracy, 6)
)

print("\nBefore split:")
print("Training clients:", train_before["client_hash_id"].nunique())
print("Testing clients:", test_before["client_hash_id"].nunique())
print("Training rows:", len(train_before))
print("Testing rows:", len(test_before))

print("\nAfter split:")
print(
    "Training dates:",
    train_after["report_date"].min(),
    "to",
    train_after["report_date"].max()
)

print(
    "Testing dates:",
    test_after["report_date"].min(),
    "to",
    test_after["report_date"].max()
)

print("Training rows:", len(train_after))
print("Testing rows:", len(test_after))


Rows used: 3611061
Features: ['gsc_impressions', 'gsc_clicks', 'ctr_pct']
Target: baseline_target

Before / After comparison:


,Validation,Precision,Accuracy
0,Week-5 client-grouped,0.934912,0.986085
1,ML-09 time-aware,0.938693,0.987124



Measured changes:
Precision change: 0.003781
Accuracy change: 0.001039

Before split:
Training clients: 37
Testing clients: 10
Training rows: 2690999
Testing rows: 920062

After split:
Training dates: 2026-03-01 00:00:00 to 2026-03-24 00:00:00
Testing dates: 2026-03-25 00:00:00 to 2026-03-31 00:00:00
Training rows: 2736046
Testing rows: 875015


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit: I reviewed the final model features to check whether any feature directly contains the target, future information, or an identifier that could allow the model to memorize records. The final feature set uses gsc_impressions, gsc_clicks, CTR, and gsc_avg_position, which are search-performance signals from the observed data window. I excluded client_hash_id and content_hash_id from the model features because they are identifiers rather than meaningful predictive signals. I also excluded future-window information and label-derived information. Based on this review, I did not identify an obvious direct leakage path in the final feature set. This is an audit of the feature definitions and should be treated as evidence supporting the model setup, not proof that every possible source of leakage has been eliminated.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-09 - Question 3: Leakage audit

features = [
    "gsc_impressions",
    "gsc_clicks",
    "CTR",
    "gsc_avg_position"
]

excluded = [
    "client_hash_id",
    "content_hash_id",
    "future-window information",
    "label-derived information"
]

print("Final model features:")
for feature in features:
    print("-", feature)

print("\nExcluded from model features:")
for item in excluded:
    print("-", item)

# Check that identifiers are not included in the feature set
assert "client_hash_id" not in features
assert "content_hash_id" not in features

# Check that all expected features are present
for feature in features:
    assert feature in df.columns, f"Missing feature: {feature}"

print("\nLeakage checks passed.")
print("No client/content identifiers are used as model features.")
print("No future-window or label-derived features are included.")


Final model features:
- gsc_impressions
- gsc_clicks
- CTR
- gsc_avg_position

Excluded from model features:
- client_hash_id
- content_hash_id
- future-window information
- label-derived information

Leakage checks passed.
No client/content identifiers are used as model features.
No future-window or label-derived features are included.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim: The Logistic Regression model can accurately identify content that is likely to decline and can help prioritize which content should be reviewed.

Rewritten claim: The Logistic Regression model showed measured predictive performance on the evaluated data and provided a directional signal for identifying content that may need further review. These results can be used as decision-support for prioritization, but they do not prove that the model will generalize equally well to every client or future reporting period. The model should therefore be treated as an observed and measured signal rather than a guarantee of future content performance.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 - Question 4: Claim rewrite

original_claim = (
    "The Logistic Regression model can accurately identify content "
    "that is likely to decline and help prioritize content for review."
)

safe_claim = (
    "The Logistic Regression model showed measured predictive performance "
    "on the evaluated data and provided a directional signal for identifying "
    "content that may need further review. The result is decision-support "
    "rather than a guarantee of future performance or generalization to every client."
)

print("Original claim:")
print(original_claim)

print("\nSafe rewritten claim:")
print(safe_claim)


Original claim:
The Logistic Regression model can accurately identify content that is likely to decline and help prioritize content for review.

Safe rewritten claim:
The Logistic Regression model showed measured predictive performance on the evaluated data and provided a directional signal for identifying content that may need further review. The result is decision-support rather than a guarantee of future performance or generalization to every client.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.